In [8]:
%load_ext autoreload
%autoreload 2

import sys
import os
import math
from importlib import reload

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model, spectra_manager, constants, population, concat

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import mars.population.transform as transform
from mars.population.contexts import Context, SummedContext
from mars import spin_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from mars.population.tr_utils import EvolutionMatrix, EvolutionSuper

In [11]:
dtype = torch.float64

In [14]:
res_energies = torch.tensor([mars.constants.unit_converter(-2, "K_to_Hz"), mars.constants.unit_converter(2, "K_to_Hz")], dtype=dtype)
temp = torch.tensor(1.0, dtype=dtype)

free_probs = torch.tensor([[0.0, 10.0], [20.0, 0.0]], dtype=dtype)
driven_probs = torch.tensor([[0.0, 10.0], [20.0, 0.0]], dtype=dtype)
evo_matrix = EvolutionMatrix(res_energies)

out = evo_matrix(temp, None , driven_probs)

In [22]:
res_energies.unsqueeze(-2)

tensor([[-4.1673e+10,  4.1673e+10]], dtype=torch.float64)

In [19]:
evo_matrix.energy_diff

tensor([[ 0.,  4.],
        [-4.,  0.]], dtype=torch.float64)

In [48]:
def test_detailed_balance_two_level():
    """Verify detailed balance condition for 2-level system"""
    dtype = torch.float64
    E_K = torch.tensor([-2.0, 2.0], dtype=dtype)
    res_energies = mars.constants.unit_converter(E_K, "K_to_Hz")
    
    temp = torch.tensor(1.0, dtype=dtype)
    
    # Symmetric mean rates: k'_01 = k'_10 = 10 s⁻¹
    free_probs = torch.tensor([[0.0, 10.0], 
                               [10.0, 0.0]], dtype=dtype)
    
    evo = EvolutionMatrix(res_energies, symmetry_probs=True)
    K = evo(temp, free_probs=free_probs)
    
    # 1. Column sums must be zero (probability conservation)
    col_sums = K.sum(dim=-2)
    assert torch.allclose(col_sums, torch.zeros_like(col_sums), atol=1e-10), \
        f"Column sums not zero: {col_sums}"
    
    # 2. Detailed balance: K[i,j]/K[j,i] = exp(-(E_i - E_j)/kT)
    ΔE_01 = E_K[0] - E_K[1]  # -4 K
    expected_ratio_01 = torch.exp(-ΔE_01 / temp)  # exp(-(-4)/1) = exp(4) ≈ 54.6
    
    # K[0,1] = rate FROM 1 TO 0 (downward, should be large)
    # K[1,0] = rate FROM 0 TO 1 (upward, should be small)
    ratio_01 = K[0, 1] / K[1, 0]
    assert torch.allclose(ratio_01, expected_ratio_01, rtol=1e-5), \
        f"Detailed balance violated: got {ratio_01}, expected {expected_ratio_01}"
    
    # 3. Equilibrium distribution must be null eigenvector: K @ n_eq = 0
    n_eq = torch.exp(-E_K / temp)
    n_eq = n_eq / n_eq.sum()  # Normalize
    dn_dt = K @ n_eq
    assert torch.allclose(dn_dt, torch.zeros_like(dn_dt), atol=1e-10), \
        f"Equilibrium not stationary: {dn_dt}"

In [49]:
def test_temperature_dependence():
    """Verify rates become symmetric at high temperature"""
    dtype = torch.float64
    res_energies = mars.constants.unit_converter(
        torch.tensor([0.0, 10.0], dtype=dtype), "K_to_Hz"
    )
    
    free_probs = torch.tensor([[0.0, 5.0], [5.0, 0.0]], dtype=dtype)
    evo = EvolutionMatrix(res_energies, symmetry_probs=True)
    
    # Low T: downward rate >> upward rate
    K_low = evo(torch.tensor(1.0, dtype=dtype), free_probs=free_probs)
    assert K_low[0, 1] > 10 * K_low[1, 0]  # 0←1 much faster than 1←0
    
    # High T: rates become symmetric
    K_high = evo(torch.tensor(10000.0, dtype=dtype), free_probs=free_probs)
    ratio = K_high[0, 1] / K_high[1, 0]
    assert torch.abs(ratio - 1.0) < 0.01, f"High-T asymmetry: {ratio}"


In [50]:
def test_superoperator_population_block():
    """Verify EvolutionSuper correctly scales population-population block"""
    dtype = torch.float64
    res_energies = mars.constants.unit_converter(
        torch.tensor([-1.0, 1.0], dtype=dtype), "K_to_Hz"
    )
    
    # Create relaxation superoperator with ONLY population transfers
    # R[pop_i, pop_j] = rate from j to i (same as K matrix)
    free_superop = torch.zeros(4, 4, dtype=dtype)
    free_superop[0, 3] = 5.0  # |0><0| ← |1><1|  (downward)
    free_superop[3, 0] = 0.1  # |1><1| ← |0><0|  (upward)
    
    # Hamiltonian (doesn't affect population block in eigenbasis)
    H = torch.diag(res_energies)
    
    evo_super = EvolutionSuper(res_energies, symmetry_probs=True)
    R = evo_super(temp=torch.tensor(1.0, dtype=dtype), 
                  H=H.unsqueeze(0), 
                  free_superop=free_superop.unsqueeze(0))
    
    R = R.squeeze(0)
    
    # Extract population block (indices 0 and 3 for 2-level system)
    pop_block = R[[0, 3]][:, [0, 3]]
    
    # Should satisfy detailed balance with energies
    E_K = torch.tensor([-1.0, 1.0], dtype=dtype)
    expected_ratio = torch.exp(-(E_K[0] - E_K[1]) / 1.0)  # exp(-(-2)) = exp(2)
    actual_ratio = pop_block[0, 1] / pop_block[1, 0]  # Rate 0←1 / 1←0
    
    assert torch.allclose(actual_ratio, expected_ratio.to(torch.complex128), rtol=1e-5), \
        f"Superoperator detailed balance failed: {actual_ratio} vs {expected_ratio}"

In [51]:
test_detailed_balance_two_level()
test_temperature_dependence()
test_superoperator_population_block()